In [7]:
import pandas as pd
import re

# 1. LOAD DATA
topik = pd.read_csv(r'C:\Users\Liza\Documents\Kerja Praktik FIXX\Emilia-dataset-evaluation\berttopic\exp-5-mpnet-base-v2\bertopic_results(5) copy.csv', sep=';', on_bad_lines='skip') 
# topik = pd.read_csv(r'C:\Users\Liza\Documents\Kerja Praktik FIXX\Emilia-dataset-evaluation\berttopic\exp 7-bert\after_reassign_indobert_clean.csv', sep=';', on_bad_lines='skip') 
dataset = pd.read_csv(r'C:\Users\Liza\Documents\Kerja Praktik FIXX\Emilia-dataset-evaluation\dataset\combined_dataset_all.csv', sep=';', on_bad_lines='skip')

# 2. PEMBERSIHAN KHUSUS KOLOM JAWABAN (Hapus Tanda Baca & Spasi)
def clean_answer_only(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)  # Hapus tanda baca
    text = re.sub(r'\s+', '', text)     # Hapus semua spasi
    return text

# Buat kunci pencocokan di kedua dataframe
topik['answer_key'] = topik['answer'].apply(clean_answer_only)
dataset['answer_key'] = dataset['answer'].apply(clean_answer_only)

# Ambil versi unik dari dataset acuan agar hitungannya murni
dataset_unik_ans = dataset.drop_duplicates(subset=['answer_key'])

# =========================================================================
# 3. JODOHKAN DATASET DENGAN TOPIK BERDASARKAN JAWABAN (Inner Join)
# =========================================================================
df_match_by_ans = pd.merge(
    dataset_unik_ans[['question', 'answer', 'answer_key']], 
    topik[['answer_key', 'Topic', 'Name']], 
    on='answer_key', 
    how='inner'
)

# Buang duplikat hasil merge jika ada baris dari file topik yang double
df_match_by_ans = df_match_by_ans.drop_duplicates(subset=['answer_key'], keep='first')

print("=== 1. HASIL PENCOCOKAN BERDASARKAN JAWABAN SAJA ===")
print(f"Jumlah data yang MATCH : {len(df_match_by_ans)} baris")
print("-" * 60)

# =========================================================================
# 4. TAMPILKAN TOPIK DAN JUMLAHNYA DARI DATA HASIL COCOK JAWABAN
# =========================================================================
print("=== 2. SEBARAN TOPIK DAN JUMLAH DATA ===")
total_topik_unik = df_match_by_ans['Topic'].nunique()
print(f"Total seluruh topik unik yang terbentuk: {total_topik_unik} topik\n")

# Hitung jumlah data per ID Topik
distribusi_topik = df_match_by_ans['Topic'].value_counts().reset_index()
distribusi_topik.columns = ['ID Topik', 'Jumlah Data']

print(distribusi_topik.to_string(index=False))
print("-" * 60)

=== 1. HASIL PENCOCOKAN BERDASARKAN JAWABAN SAJA ===
Jumlah data yang MATCH : 7488 baris
------------------------------------------------------------
=== 2. SEBARAN TOPIK DAN JUMLAH DATA ===
Total seluruh topik unik yang terbentuk: 27 topik

 ID Topik  Jumlah Data
        0         3781
       -1          720
        1          679
        2          267
        3          211
        4          198
        6          169
        5          149
        7          138
       10          120
        9          112
        8          107
       11          103
       14           78
       15           68
       12           67
       22           67
       13           62
       25           51
       17           51
       19           51
       16           49
       20           45
       18           41
       24           37
       21           37
       23           30
------------------------------------------------------------


In [13]:
# =========================================================================
# MENAMPILKAN DATA DUPLIKAT DAN INDEKSNYA DI FILE TOPIK
# =========================================================================
# keep=False artinya semua baris yang kembar akan ditandai sebagai True
duplikat_mask = topik.duplicated(subset=['question', 'answer'], keep=False)
df_duplikat = topik[duplikat_mask]

print("\n=== DATA DUPLIKAT DI FILE TOPIK ===")
if len(df_duplikat) > 0:
    print(f"Ditemukan {len(df_duplikat)} baris yang saling menduplikat satu sama lain.")
    print("Berikut adalah data beserta nomor indeks aslinya:\n")
    
    # Menampilkan indeks, question, dan answer
    for idx, row in df_duplikat.iterrows():
        print(f"[Indeks: {idx}]")
        print(f"Q: {row['question']}")
        print(f"A: {row['answer']}")
        print("-" * 30)
else:
    print("Tidak ditemukan data duplikat.")


=== DATA DUPLIKAT DI FILE TOPIK ===
Ditemukan 24 baris yang saling menduplikat satu sama lain.
Berikut adalah data beserta nomor indeks aslinya:

[Indeks: 49]
Q: Aku marah karena tetangga parkir sembarangan. Rasanya pengen protes. Gimana caranya biar nggak emosi?
A: Parkir sembarangan emang bikin kesel, ya. Coba pas marah, tarik napas dalam, lalu tanya: ‘Fakta apa? Mobilnya salah parkir, tapi nggak berarti sengaja.’ Daripada protes, coba ngobrol santai sama tetangga buat cari solusi, atau fokus ke hal lain yang bikin tenang. Tulis tiga hal yang bikin kamu bersyukur—mungkin rumahmu. Langkah ini bantu kepala dingin. Yuk, coba ngobrol ramah sama tetangga besok!
------------------------------
[Indeks: 51]
Q: Aku marah karena tetangga parkir sembarangan. Rasanya pengen protes. Gimana caranya biar nggak emosi?
A: Parkir sembarangan emang bikin kesel, ya. Coba pas marah, tarik napas dalam, lalu tanya: ‘Fakta apa? Mobilnya salah parkir, tapi nggak berarti sengaja.’ Daripada protes, coba ngobr

In [14]:
print("=== TEKS UTUH DI TOPIK ===")
print("Question:", repr(hanya_di_topik['question'].values[0]))
print("Answer  :", repr(hanya_di_topik['answer'].values[0]))

print("\n=== TEKS UTUH DI DATASET ===")
print("Question:", repr(hanya_di_dataset['question'].values[0]))
print("Answer  :", repr(hanya_di_dataset['answer'].values[0]))

=== TEKS UTUH DI TOPIK ===
Question: 'Aku ngerasa nggak ada yang ngerti apa yang aku rasain. Kayaknya aku sendirian banget. Apa yang bisa aku lakuin?'
Answer  : 'Rasa sendirian itu pasti berat, ya. Aku di sini buat dengerin, dan aku yakin ada orang lain yang juga bisa ngerti kalau kamu kasih kesempatan. Mungkin kamu bisa coba ceritain sedikit tentang apa yang kamu rasain ke seseorang yang kamu percaya, atau bahkan ke aku dulu. Kadang, ngomongin perasaan bisa bikin beban terasa lebih ringan. Mau coba ceritain apa yang lagi bikin kamu ngerasa gitu?'

=== TEKS UTUH DI DATASET ===
Question: 'Aku ngerasa nggk ada yang ngerti apa yang aku rasain. Kayaknya aku sendirian banget. Apa yang bisa aku lakuin?'
Answer  : 'Rasa sendirian itu pasti berat, ya. Aku di sini buat dengerin, dan aku yakin ada orang lain yang juga bisa ngerti kalau kamu kasih kesempatan. Mungkin kamu bisa coba ceritain sedikit tentang apa yang kamu rasain ke seseorang yang kamu percaya, atau bahkan ke aku dulu. Kadang, ngomo

In [15]:
topik = pd.read_csv(r'C:\Users\Liza\Documents\Kerja Praktik FIXX\Emilia-dataset-evaluation\berttopic\exp 7-bert\after_reassign_indobert copy.csv', sep=';', on_bad_lines='skip') 
topik.info()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Liza\\Documents\\Kerja Praktik FIXX\\Emilia-dataset-evaluation\\berttopic\\exp 7-bert\\after_reassign_indobert copy.csv'

In [ ]:
dataset = pd.read_csv(r'C:\Users\Liza\Documents\Kerja Praktik FIXX\Emilia-dataset-evaluation\dataset\combined_dataset_all.csv', sep=';', on_bad_lines='skip')
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7488 entries, 0 to 7487
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  7488 non-null   object
 1   answer    7488 non-null   object
 2   from      7488 non-null   object
 3   source    7488 non-null   object
 4   folder    7488 non-null   object
dtypes: object(5)
memory usage: 292.6+ KB


In [ ]:
topik = pd.read_csv(r'exp-5-mpnet-base-v2\bertopic_results(5).csv', sep=';', on_bad_lines='skip') 
topik.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   question               7500 non-null   object 
 1   answer                 7500 non-null   object 
 2   from                   7500 non-null   object 
 3   filename               7500 non-null   object 
 4   document_preprocessed  7500 non-null   object 
 5   Topic                  7500 non-null   int64  
 6   Probability            7500 non-null   float64
 7   Top_n_words            7500 non-null   object 
 8   Name                   7500 non-null   object 
 9   Document               7500 non-null   object 
dtypes: float64(1), int64(1), object(8)
memory usage: 586.1+ KB


In [ ]:
df_duplikat = topik[topik.duplicated(subset=['question', 'answer'], keep=False)]

df_duplikat_sorted = df_duplikat.sort_values(by=['question', 'answer'])

print(f"Ditemukan {len(df_duplikat_sorted)} baris yang terduplikasi.\n")

df_duplikat_sorted

Ditemukan 24 baris yang terduplikasi.



,question,answer,from,filename,document_preprocessed,Topic,Probability,Top_n_words,Name,Document
6435,Aku kesal karena temen bilang aku nggak asik. ...,"Komentar temen bisa bikin nggak enak, ya. Tapi...",GROK,A1_GROK_FilosofiTeras124-148_53.json,kesal temen serang sakit hati komentar temen e...,18,0.384429,temen - diri - medsos - salah - kecil - kritik...,18_medsos_fakta_iklan_ramah,kesal temen serang sakit hati komentar temen e...
6437,Aku kesal karena temen bilang aku nggak asik. ...,"Komentar temen bisa bikin nggak enak, ya. Tapi...",GROK,A1_GROK_FilosofiTeras124-148_53.json,kesal temen serang sakit hati komentar temen e...,18,0.386856,temen - diri - medsos - salah - kecil - kritik...,18_medsos_fakta_iklan_ramah,kesal temen serang sakit hati komentar temen e...
49,Aku marah karena tetangga parkir sembarangan. ...,"Parkir sembarangan emang bikin kesel, ya. Coba...",GROK,A1_GROK_FilosofiTeras124-148_53.json,marah tetangga parkir sembarang protes emosi p...,14,0.249160,sepi - sendiri - kecil - sosial - temen - nyam...,14_tetangga_sosial_sepi_teman,marah tetangga parkir sembarang protes emosi p...
51,Aku marah karena tetangga parkir sembarangan. ...,"Parkir sembarangan emang bikin kesel, ya. Coba...",GROK,A1_GROK_FilosofiTeras124-148_53.json,marah tetangga parkir sembarang protes emosi p...,14,0.249377,sepi - sendiri - kecil - sosial - temen - nyam...,14_tetangga_sosial_sepi_teman,marah tetangga parkir sembarang protes emosi p...
3766,"Aku merasa pendiam dan nggak menarik, makanya ...","Nggak ada yang salah dengan sifat pendiam, itu...",GROK,D6_GROK_BeraniTidakDisukaiW86-96_15.json,diam makanya susah temen salah salah sifat dia...,0,0.044288,diri - takut - sendiri - kecil - salah - gagal...,-1_hidup_kecil_diri_sendiri,diam makanya susah temen salah salah sifat dia...
3767,"Aku merasa pendiam dan nggak menarik, makanya ...","Nggak ada yang salah dengan sifat pendiam, itu...",GROK,D6_GROK_BeraniTidakDisukaiW86-96_15.json,diam makanya susah temen salah salah sifat dia...,0,0.043766,diri - takut - sendiri - kecil - salah - gagal...,-1_hidup_kecil_diri_sendiri,diam makanya susah temen salah salah sifat dia...
1030,Aku ngerasa kurang gara-gara nggak punya libur...,"Pengen liburan mewah bisa bikin merasa kurang,...",GROK,A2_GROK_FilosofiTeras159-182_60.json,ngerasa kurang gara gara libur mewah cukup lib...,-1,0.169356,outlier,-1_hidup_kecil_diri_sendiri,ngerasa kurang gara gara libur mewah cukup lib...
1035,Aku ngerasa kurang gara-gara nggak punya libur...,"Pengen liburan mewah bisa bikin merasa kurang,...",GROK,A2_GROK_FilosofiTeras159-182_60.json,ngerasa kurang gara gara libur mewah cukup lib...,-1,0.166511,outlier,-1_hidup_kecil_diri_sendiri,ngerasa kurang gara gara libur mewah cukup lib...
154,Aku ngerasa temenku cuma deket kalau aku punya...,Temen yang cuma deket pas ada waktu bisa bikin...,GROK,A2_GROK_FilosofiTeras191-227_54.json,ngerasa temenku deket luang stuck teman temen ...,6,0.207847,temen - nikah - bahagia - temenku - hancur - d...,6_temen_nikah_bahagia_temenku,ngerasa temenku deket luang stuck teman temen ...
157,Aku ngerasa temenku cuma deket kalau aku punya...,Temen yang cuma deket pas ada waktu bisa bikin...,GROK,A2_GROK_FilosofiTeras191-227_54.json,ngerasa temenku deket luang stuck teman temen ...,6,0.207920,temen - nikah - bahagia - temenku - hancur - d...,6_temen_nikah_bahagia_temenku,ngerasa temenku deket luang stuck teman temen ...
